# Classifier MG-Positive Window Distributions

For each classifier and feature-window size, compare the distribution of MG-positive windows per HG/MG text. A window is MG-positive when its classifier score is at least 0.5.

In [ ]:
from __future__ import annotations

import os
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from tqdm.auto import tqdm

from notebooks.utils.classification_experiment_utils import discover_model_artifacts
from notebooks.utils.classifier_window_distribution_utils import (
    aggregate_window_classifications,
    plot_window_distribution,
)
from notebooks.utils.feature_aggregation_utils import (
    FeatureSet,
    ensure_prepared_partition,
    load_prepared_partition,
    prepared_partition_path,
)
from notebooks.utils.model_training_utils import (
    load_model_artifact,
    load_model_metadata,
    score_feature_vector_model,
)

DATA_ROOT = Path(os.environ.get('TEXT_DETECTION_DATA', 'D:/Projects/master_diploma/text-detection-data'))
PROCESSED_ROOT = DATA_ROOT / 'processed'
MODELS_ROOT = DATA_ROOT / 'experiment-results' / 'classification' / 'models'
SCORER_LLM = 'gemma_3_270m_it'
DATASET = 'name_that_llm_train'
# Kaggle
# DATASET = "kaggle_train"
# BBC
# DATASET = "bbc_news_content"
MIN_TOKEN_COUNT = 250
NUM_TEXTS = 200
RUN_ID = 'feature-vector-classification-2026-09-25-logreg-random-forest-max600-min100'
TRAINING_COMPOSITION = 'Kaggle+MAGE+Outfox'
MODEL_FAMILIES = ('logistic_regression', 'random_forest')
WINDOWS = (32, 64, 128)
STRIDE = 16
FEATURE_SET = FeatureSet('simple_similarity', (
    'mean_candidate_similarity__mean', 'mean_candidate_similarity__std',
    'top20_coverage__mean', 'top20_coverage__std',
    'observed_logprob__mean', 'observed_logprob__std',
    'mean_candidate_logprob_delta__mean', 'mean_candidate_logprob_delta__std',
))
print({'dataset': DATASET, 'scorer_llm': SCORER_LLM, 'min_token_count': MIN_TOKEN_COUNT, 'num_texts_per_split': NUM_TEXTS})

## Select HG/MG cohorts

The token lower bound is applied to whole-text token counts stored by the scoring LLM. Texts are selected independently per split.

In [ ]:
def load_text_metadata(split: str) -> pd.DataFrame:
    paths = sorted((PROCESSED_ROOT / f'{DATASET}__{SCORER_LLM}' / split / 'data').glob('*.parquet'))
    if not paths:
        raise FileNotFoundError(f'no processed files found for {DATASET}/{SCORER_LLM}/{split}')
    metadata = pd.concat((pd.read_parquet(path, columns=['id', 'tokens']) for path in paths), ignore_index=True)
    metadata['id'] = metadata['id'].astype(str).str.strip()
    metadata = metadata.drop_duplicates('id').copy()
    metadata['token_count'] = metadata['tokens'].map(len)
    metadata = metadata.loc[metadata['token_count'] >= MIN_TOKEN_COUNT].copy()
    metadata = metadata.sort_values('id', key=lambda values: values.map(int)).reset_index(drop=True)
    if len(metadata) < NUM_TEXTS:
        raise ValueError(f'{DATASET}/{split} has only {len(metadata)} texts with token_count >= {MIN_TOKEN_COUNT}; expected {NUM_TEXTS}')
    return metadata.iloc[:NUM_TEXTS][['id', 'token_count']].copy()

selected_metadata = {}
unavailable_splits = {}
for split in ('HG', 'MG'):
    try:
        selected_metadata[split] = load_text_metadata(split)
    except FileNotFoundError as error:
        unavailable_splits[split] = str(error)

if unavailable_splits:
    print(f'Skipping unavailable splits: {unavailable_splits}')
if not selected_metadata:
    raise ValueError(f'no HG or MG processed data is available for {DATASET}/{SCORER_LLM}')
pd.DataFrame({split: {'text_count': len(frame), 'min_token_count': frame['token_count'].min()} for split, frame in selected_metadata.items()}).T

## Score every classifier/window configuration

Each prepared feature row is one segment window. Scores are retained at segment level before aggregation by text.

In [ ]:
def expected_prepared_metadata(split: str, window: int) -> dict[str, object]:
    return {
        'scorer_llm': SCORER_LLM,
        'dataset': DATASET,
        'split': split,
        'feature_set': FEATURE_SET.name,
        'feature_columns': list(FEATURE_SET.source_columns),
        'window_configuration': str(window),
        'schema_version': 1,
        'stride': STRIDE,
    }

def load_selected_partition(split: str, window: int) -> pd.DataFrame:
    status = ensure_prepared_partition(
        DATA_ROOT, PROCESSED_ROOT, scorer_llm=SCORER_LLM, dataset=DATASET,
        split=split, feature_set=FEATURE_SET, window_configuration=window, stride=STRIDE,
    )
    if status.get('status') != 'available':
        raise RuntimeError(f'feature preparation unavailable for {split}/{window}: {status}')
    path = prepared_partition_path(
        DATA_ROOT, scorer_llm=SCORER_LLM, dataset=DATASET, split=split,
        feature_set=FEATURE_SET, window_configuration=window,
    )
    frame = load_prepared_partition(path, expected=expected_prepared_metadata(split, window)).copy()
    frame['text_id'] = frame['text_id'].astype(str).str.strip()
    selected_ids = set(selected_metadata[split]['id'])
    frame = frame.loc[frame['text_id'].isin(selected_ids)].copy()
    missing_ids = selected_ids.difference(frame['text_id'].unique())
    if missing_ids:
        raise ValueError(f'{split}/{window} is missing selected text IDs: {sorted(missing_ids)[:5]}')
    return frame

def find_model_artifact(model_family: str, window: int) -> Path:
    matches = discover_model_artifacts(MODELS_ROOT, filters={
        'run_id': RUN_ID, 'model_family': model_family, 'scorer_llm': SCORER_LLM,
        'feature_set': FEATURE_SET.name, 'training_regime': TRAINING_COMPOSITION,
        'training_window': str(window),
    })
    if len(matches) != 1:
        raise ValueError(f'expected one artifact for {model_family}/{window}, found {matches}')
    return matches[0]

def score_configuration(split: str, window: int, model_family: str) -> pd.DataFrame:
    artifact_path = find_model_artifact(model_family, window)
    metadata = load_model_metadata(artifact_path, expected={
        'run_id': RUN_ID, 'model_family': model_family, 'scorer_llm': SCORER_LLM,
        'feature_set': FEATURE_SET.name, 'training_regime': TRAINING_COMPOSITION,
        'training_window': str(window),
    })
    if metadata['feature_columns'] != list(FEATURE_SET.source_columns):
        raise ValueError(f'artifact feature schema mismatch: {artifact_path}')
    fitted = load_model_artifact(artifact_path, expected={'feature_columns': list(FEATURE_SET.source_columns)}, trust_artifact=True)
    frame = load_selected_partition(split, window)
    columns = ['split', 'text_id', 'label', *FEATURE_SET.source_columns]
    scored = score_feature_vector_model(fitted, frame[columns])
    return scored.assign(split=split, model_family=model_family, window_configuration=window)

segment_scores = []
for split in selected_metadata:
    for window in tqdm(WINDOWS, desc=f'{split} windows'):
        for model_family in MODEL_FAMILIES:
            segment_scores.append(score_configuration(split, window, model_family))
segment_scores = pd.concat(segment_scores, ignore_index=True)
window_metrics = aggregate_window_classifications(segment_scores)
display(window_metrics.head())
display(window_metrics.groupby(['split', 'model_family', 'window_configuration']).size().rename('text_count').reset_index())

## Distributions

In [ ]:
figure, axes = plot_window_distribution(window_metrics, 'mg_window_count')
figure.suptitle('MG-positive windows per text: absolute count', y=1.03)
plt.show()

figure, axes = plot_window_distribution(window_metrics, 'mg_window_percent')
figure.suptitle('MG-positive windows per text: percentage', y=1.03)
plt.show()

In [ ]:
import seaborn as sns

def plot_smoothed_window_distribution(value_column: str, title: str) -> None:
    configurations = list(
        window_metrics[['model_family', 'window_configuration']]
        .drop_duplicates()
        .sort_values(['model_family', 'window_configuration'])
        .itertuples(index=False, name=None)
    )
    figure, axis_grid = plt.subplots(1, len(configurations), figsize=(5 * len(configurations), 4), squeeze=False)
    for axis, (model_family, window_configuration) in zip(axis_grid.ravel(), configurations):
        subset = window_metrics.loc[
            window_metrics['model_family'].eq(model_family)
            & window_metrics['window_configuration'].eq(window_configuration)
        ]
        for split, color in (('HG', 'tab:blue'), ('MG', 'tab:orange')):
            values = subset.loc[subset['split'].eq(split), value_column]
            if values.nunique() > 1:
                sns.kdeplot(values, ax=axis, color=color, label=split, cut=0, bw_adjust=0.8)
            elif not values.empty:
                axis.axvline(values.iloc[0], color=color, label=split)
        axis.set_title(f'{model_family} / {window_configuration}')
        axis.set_xlabel('MG-positive windows per text' if value_column == 'mg_window_count' else 'MG-positive windows (%)')
        axis.set_ylabel('Density')
        axis.legend(title='split')
        if value_column == 'mg_window_percent':
            axis.set_xlim(0, 100)
    figure.suptitle(title, y=1.03)
    figure.tight_layout()
    plt.show()

plot_smoothed_window_distribution('mg_window_count', 'Smoothed MG-positive windows per text: absolute count')
plot_smoothed_window_distribution('mg_window_percent', 'Smoothed MG-positive windows per text: percentage')

In [ ]:
DATASETS = (
    'kaggle_train',
    # 'mage_train',
    'bbc_news_content',
    'outfox_chatgpt_train',
    'semeval2024_task8_subtask_c_train',
    'name_that_llm_train',
)
COMBINED_NUM_TEXTS = NUM_TEXTS

def select_dataset_metadata(dataset: str, split: str) -> pd.DataFrame:
    paths = sorted((PROCESSED_ROOT / f'{dataset}__{SCORER_LLM}' / split / 'data').glob('*.parquet'))
    if not paths:
        raise FileNotFoundError(f'no processed files found for {dataset}/{SCORER_LLM}/{split}')
    metadata = pd.concat((pd.read_parquet(path, columns=['id', 'tokens']) for path in paths), ignore_index=True)
    metadata['id'] = metadata['id'].astype(str).str.strip()
    metadata = metadata.drop_duplicates('id').copy()
    metadata['token_count'] = metadata['tokens'].map(len)
    metadata = metadata.loc[metadata['token_count'] >= MIN_TOKEN_COUNT].copy()
    metadata = metadata.sort_values('id', key=lambda values: values.map(int)).reset_index(drop=True)
    if len(metadata) < COMBINED_NUM_TEXTS:
        raise ValueError(f'{dataset}/{split} has only {len(metadata)} texts with token_count >= {MIN_TOKEN_COUNT}; expected {COMBINED_NUM_TEXTS}')
    return metadata.iloc[:COMBINED_NUM_TEXTS][['id', 'token_count']].copy()

def score_dataset(dataset: str, selected: dict[str, pd.DataFrame]) -> pd.DataFrame:
    frames = []
    for split, metadata in selected.items():
        selected_ids = set(metadata['id'])
        for window in tqdm(WINDOWS, desc=f'{dataset} {split} windows'):
            expected = {
                'scorer_llm': SCORER_LLM, 'dataset': dataset, 'split': split,
                'feature_set': FEATURE_SET.name, 'feature_columns': list(FEATURE_SET.source_columns),
                'window_configuration': str(window), 'schema_version': 1, 'stride': STRIDE,
            }
            status = ensure_prepared_partition(
                DATA_ROOT, PROCESSED_ROOT, scorer_llm=SCORER_LLM, dataset=dataset,
                split=split, feature_set=FEATURE_SET, window_configuration=window, stride=STRIDE,
            )
            if status.get('status') != 'available':
                raise RuntimeError(f'feature preparation unavailable for {dataset}/{split}/{window}: {status}')
            path = prepared_partition_path(
                DATA_ROOT, scorer_llm=SCORER_LLM, dataset=dataset, split=split,
                feature_set=FEATURE_SET, window_configuration=window,
            )
            partition = load_prepared_partition(path, expected=expected).copy()
            partition['text_id'] = partition['text_id'].astype(str).str.strip()
            partition = partition.loc[partition['text_id'].isin(selected_ids)].copy()
            missing_ids = selected_ids.difference(partition['text_id'].unique())
            if missing_ids:
                raise ValueError(f'{dataset}/{split}/{window} is missing selected text IDs: {sorted(missing_ids)[:5]}')
            columns = ['split', 'text_id', 'label', *FEATURE_SET.source_columns]
            for model_family in MODEL_FAMILIES:
                artifact_path = find_model_artifact(model_family, window)
                fitted = load_model_artifact(artifact_path, expected={'feature_columns': list(FEATURE_SET.source_columns)}, trust_artifact=True)
                scored = score_feature_vector_model(fitted, partition[columns])
                frames.append(scored.assign(split=split, model_family=model_family, window_configuration=window))
    return aggregate_window_classifications(pd.concat(frames, ignore_index=True)).assign(dataset=dataset)

combined_frames = []
for dataset in DATASETS:
    selected = {}
    for split in ('HG', 'MG'):
        try:
            selected[split] = select_dataset_metadata(dataset, split)
        except FileNotFoundError as error:
            print(f'Skipping unavailable split: {error}')
    if not selected:
        raise ValueError(f'no HG or MG processed data is available for {dataset}/{SCORER_LLM}')
    combined_frames.append(score_dataset(dataset, selected))

combined_window_metrics = pd.concat(combined_frames, ignore_index=True)

dataset_colors = dict(zip(DATASETS, sns.color_palette('tab10', n_colors=len(DATASETS))))
split_styles = {'HG': '-', 'MG': '--'}
for value_column, title, xlabel in (
    ('mg_window_count', 'Combined datasets: MG-positive windows per text', 'MG-positive windows per text'),
    ('mg_window_percent', 'Combined datasets: MG-positive windows percentage', 'MG-positive windows (%)'),
):
    configurations = list(combined_window_metrics[['model_family', 'window_configuration']].drop_duplicates().sort_values(['model_family', 'window_configuration']).itertuples(index=False, name=None))
    figure, axes = plt.subplots(1, len(configurations), figsize=(5 * len(configurations), 4), squeeze=False)
    for axis, (model_family, window_configuration) in zip(axes.ravel(), configurations):
        subset = combined_window_metrics.loc[
            combined_window_metrics['model_family'].eq(model_family)
            & combined_window_metrics['window_configuration'].eq(window_configuration)
        ]
        for (dataset, split), values in subset.groupby(['dataset', 'split'], sort=True)[value_column]:
            label = f'{dataset} / {split}'
            if values.nunique() > 1:
                sns.kdeplot(values, ax=axis, color=dataset_colors[dataset], linestyle=split_styles[split], label=label, cut=0, bw_adjust=0.8)
            elif not values.empty:
                axis.axvline(values.iloc[0], color=dataset_colors[dataset], linestyle=split_styles[split], label=label)
        axis.set_title(f'{model_family} / {window_configuration}')
        axis.set_xlabel(xlabel)
        axis.set_ylabel('Density')
        if value_column == 'mg_window_percent':
            axis.set_xlim(0, 100)
        axis.legend(fontsize='small')
    figure.suptitle(title, y=1.03)
    figure.tight_layout()
    plt.show()

In [ ]:
EXPECTED_SOURCE_LLM_COUNT = 18
source_metadata = pd.read_csv(
    DATA_ROOT / 'raw' / 'name-that-llm' / 'name_that_llm_train_MG.csv',
    usecols=['id', 'source_llm'],
    dtype={'id': str, 'source_llm': str},
).assign(id=lambda frame: frame['id'].str.strip())

name_that_llm_window_metrics = combined_window_metrics.loc[
    combined_window_metrics['dataset'].eq('name_that_llm_train')
    & combined_window_metrics['split'].eq('MG')
].copy()
name_that_llm_window_metrics['text_id'] = name_that_llm_window_metrics['text_id'].astype(str).str.strip()
source_llm_window_metrics = name_that_llm_window_metrics.merge(
    source_metadata.rename(columns={'id': 'text_id'}),
    on='text_id',
    how='left',
    validate='many_to_one',
)
if source_llm_window_metrics['source_llm'].isna().any():
    raise ValueError('source LLM metadata is missing for one or more window rows')
if source_llm_window_metrics['source_llm'].nunique() != EXPECTED_SOURCE_LLM_COUNT:
    raise ValueError(
        f"expected {EXPECTED_SOURCE_LLM_COUNT} source LLMs, "
        f"found {source_llm_window_metrics['source_llm'].nunique()}"
    )

source_llm_colors = dict(
    zip(
        sorted(source_llm_window_metrics['source_llm'].unique()),
        sns.color_palette('tab20', n_colors=EXPECTED_SOURCE_LLM_COUNT),
    )
)
for value_column, title, xlabel in (
    ('mg_window_count', 'Name-That-LLM source LLMs: MG-positive windows per text', 'MG-positive windows per text'),
    ('mg_window_percent', 'Name-That-LLM source LLMs: MG-positive windows percentage', 'MG-positive windows (%)'),
):
    configurations = list(
        source_llm_window_metrics[['model_family', 'window_configuration']]
        .drop_duplicates()
        .sort_values(['model_family', 'window_configuration'])
        .itertuples(index=False, name=None)
    )
    figure, axes = plt.subplots(1, len(configurations), figsize=(5 * len(configurations), 4), squeeze=False)
    for axis, (model_family, window_configuration) in zip(axes.ravel(), configurations):
        subset = source_llm_window_metrics.loc[
            source_llm_window_metrics['model_family'].eq(model_family)
            & source_llm_window_metrics['window_configuration'].eq(window_configuration)
        ]
        for source_llm in sorted(source_llm_colors):
            values = subset.loc[subset['source_llm'].eq(source_llm), value_column]
            if values.nunique() > 1:
                sns.kdeplot(values, ax=axis, color=source_llm_colors[source_llm], label=source_llm, cut=0, bw_adjust=0.8)
            elif not values.empty:
                axis.axvline(values.iloc[0], color=source_llm_colors[source_llm], label=source_llm)
        axis.set_title(f'{model_family} / {window_configuration}')
        axis.set_xlabel(xlabel)
        axis.set_ylabel('Density')
        if value_column == 'mg_window_percent':
            axis.set_xlim(0, 100)
        axis.legend(title='source LLM', fontsize='small')
    figure.suptitle(title, y=1.03)
    figure.tight_layout()
    plt.show()
